In [1]:
import pandas as pd
import time
import os 
from fastai.vision.all import *

# ⏱️ Start timer
start_time = time.time()
print("⚡ IGNITING EMERGENCY FAIL-PROOF SCRIPT...")

# ---------------------------------------------------------
# STEP 1: BRUTE-FORCE PATH FINDER (Ignores missing CSVs)
# ---------------------------------------------------------
train_path = None
comp_test_path = None

for root, dirs, files in os.walk('/kaggle/input'):
    # Look directly for the 'A' folder inside the train data
    if os.path.basename(root) == 'A' and any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in files):
        train_path = Path(root).parent
    # Look directly for the test folder containing images
    if os.path.basename(root) == 'test' and any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in files):
        comp_test_path = Path(root)

if train_path is None or comp_test_path is None:
    print("🚨 CRITICAL ERROR: Could not find train or test folders.")
    exit()

print(f"📁 TRAIN FOLDER SECURED: {train_path}")
print(f"📁 TEST FOLDER SECURED: {comp_test_path}")

# ---------------------------------------------------------
# STEP 2: LOAD DATA & FIX LABELS
# ---------------------------------------------------------
def get_label(file_path):
    name = file_path.parent.name.lower()
    if 'space' in name: return 'space'
    if 'nothing' in name: return 'nothing'
    if 'del' in name or 'delete' in name: return 'del'
    return name.upper()

train_files = get_image_files(train_path)
print(f"📊 FOUND {len(train_files)} TRAINING IMAGES")

dls = ImageDataLoaders.from_path_func(
    path=".", fnames=train_files, label_func=get_label,
    valid_pct=0.1, seed=42, item_tfms=Resize(224), 
    batch_tfms=aug_transforms(do_flip=False, max_rotate=15.0, max_lighting=0.3),
    bs=128 # Fast batch size
)

# ---------------------------------------------------------
# STEP 3: TRAIN THE MODEL (RESNET50)
# ---------------------------------------------------------
epochs = 2 # 2 Epochs for MAXIMUM SPEED
print(f"🧠 Building ResNet50... Training for {epochs} epochs.")
learn = vision_learner(dls, resnet50, metrics=[accuracy]).to_fp16()
learn.fine_tune(epochs)

# ---------------------------------------------------------
# STEP 4: PREDICT ON THE NEW 26,000 TEST IMAGES
# ---------------------------------------------------------
print("🔍 Predicting on the 26,000 unseen test images...")
test_files = get_image_files(comp_test_path).sorted()

# bs=128 for very fast prediction
test_dl = learn.dls.test_dl(test_files, bs=128) 

# TTA set to 2 to save time while still double checking
preds, _ = learn.tta(dl=test_dl, n=2)

# ---------------------------------------------------------
# STEP 5: BUILD CSV FROM SCRATCH
# ---------------------------------------------------------
predicted_labels = [dls.vocab[i] for i in preds.argmax(dim=1)]

# Manually create the dataframe since the organizers didn't give us a sample
ss = pd.DataFrame({
    'image_id': [f.name for f in test_files],
    'label': predicted_labels
})

# Safety check: Only exact allowed classes
valid_classes = set("ABCDEFGHIJKLMNOPQRSTUVWXYZ") | {"space", "del", "nothing"}
ss.loc[~ss['label'].isin(valid_classes), 'label'] = 'A'

ss.to_csv('submission.csv', index=False)
print("💾 SAVED submission.csv.")

print(f"✅ FINISHED! Total time: {(time.time() - start_time)/60:.1f} minutes.")
learn.export('resnet50_final.pkl')
print("Success! Your ResNet50 model is exported.")


⚡ IGNITING EMERGENCY FAIL-PROOF SCRIPT...
📁 TRAIN FOLDER SECURED: /kaggle/input/competitions/sign-language-contest/train
📁 TEST FOLDER SECURED: /kaggle/input/competitions/sign-language-contest/test
📊 FOUND 69600 TRAINING IMAGES
🧠 Building ResNet50... Training for 2 epochs.
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 213MB/s]


epoch,train_loss,valid_loss,accuracy,time
0,0.462229,0.137029,0.954454,04:24


epoch,train_loss,valid_loss,accuracy,time
0,0.016126,0.001014,0.999713,04:52
1,0.004158,0.000085,1.000000,04:49


🔍 Predicting on the 26,000 unseen test images...


epoch,train_loss,valid_loss,accuracy,time


<div></div>

💾 SAVED submission.csv.
✅ FINISHED! Total time: 20.8 minutes.
Success! Your ResNet50 model is exported.
